In [1]:
import sys
import types
import pandas as pd
import numpy as np
import random
from sklearn import linear_model

print("Python version:", sys.version)
print("Environment successfully initialized!")

Python version: 3.14.7 (main, Aug  5 2026, 10:29:49) [Clang 21.0.0 (clang-2100.1.1.101)]
Environment successfully initialized!


**# Historical Data**

Load historical data, remove irrelevant data, and merge it to be used for model training.

First load the machine data frame from historical data.

Machines have different characteristics such as:

- capacity (how much they can produce per period),
- remaining life is the number of period before recommended maintenance according to venfor,
- cost and loss for maintenance and repair (in general the impact of repairing after failure is higher than the impact of maintaining before failure),
- etc

In this phase, we fetch the raw records of our factory machines from the public repository, strip away the bloated, irrelevant telemetry columns, and isolate the core metrics our machine learning model needs to learn from.

In [38]:
df_historical_machine = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/historical-machine.csv')
df_historical_machine.head()


,id,capacity,remaining life,cost of maintenance,maintenance loss,cost of repair,repair loss,loss per life day,production value unit
0,M-01,130,5,50,50,100,100,20,10
1,M-02,100,9,50,50,100,100,20,10
2,M-03,140,14,50,50,100,100,20,10
3,M-04,140,13,50,50,100,100,20,10
4,M-05,130,15,50,50,100,100,20,10


For the predictive algorithm training you only need a subset of these columns, so first do some cleaning.

The column 'life' represents the number of days before failure, according to the vendor's specifications.

System Architecture Insight: Think of this like pulling an uncompressed raw configuration manifest (.csv) containing global asset telemetry—things like machine capacity, vendor-estimated lifespan, repair costs, and maintenance penalties. When you run this cell, it will render a neat preview table showing your first five machines (M-01 through M-05)

In [39]:
df_historical_machine = df_historical_machine[['id', 'remaining life']]
df_historical_machine.columns = ['id', 'life']
df_historical_machine = df_historical_machine.set_index(['id'])
df_historical_machine.head()

,life
id,
M-01,5
M-02,9
M-03,14
M-04,13
M-05,15


Next, load the production for these machines.

System Architecture Insight: Right now, the raw table has too much noise (repair costs, loss percentages, etc.) that our predictive regression model doesn't care about yet. This code block acts as a filter pass:

1. It selects only the machine identifier (id) and the vendor's estimated lifespan (remaining life).  

2. It renames the column header cleanly to just life.  

3. It sets the machine id as the index key (turning the ID column into a unique primary lookup key,     similar to a primary key in a database table).  

When you run both cells [2 & 4], you should see a clean, indexed table displaying the life expectancy of your first five machines

Next, load the production for these machines.

Now that our historical machine data is loaded and cleaned, we are ready to pull in the factory's historical production numbers.

When you run this, it will load up the daily production logs for your machine fleet, showing you how much output each machine was generating day by day.

In [40]:
df_historical_production = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/historical-production.csv')
df_historical_production.head()

,machine,day,production
0,M-01,Day-01,106
1,M-01,Day-02,104
2,M-01,Day-03,130
3,M-01,Day-04,130
4,M-01,Day-05,130


The following is a chart representing the historical production for machine M-01.

df_production_m01 = df_historical_production[df_historical_production.machine == 'M-01']

alt.Chart(df_production_m01).mark_trail().encode(
    x='day',
    y='production:T',
).properties(
    width=800,
    height=300
)
ln5  og

Error loading script: Script error for: vega-embed http://requirejs.org/docs/errors.html#scripterror
Reshape the data so that it can be used in training the predictive model.
out5 og

Production will also be used as input for the predictive model as the level of production is certainly having an impact on possible failure.

df_historical_production.columns = ['id', 'day', 'production']
df_historical_production = df_historical_production.pivot(index='id', columns='day', values='production')
df_historical_production['total'] = df_historical_production.values[:, 1:].sum(1)
df_historical_production.head()
ln6 og

[pg] Now, right after cleanly loading daily production records, raw data is usually formatted in a "long" format (meaning every single row is just one machine on one specific day). But to train a machine learning model, our algorithm needs a "wide" format—where each machine is its own single row, and every day acts as its own column. [j] **IMPORTANT TALKING POINT**


Let's do the data transformation (pivoting) step.

Step-by-Step: Reshaping the Production Data
Create a new code cell in your predictive_maintenance.ipynb notebook, paste the code below, and run it (Shift + Enter):

# Standardize column names just in case, then reshape into a wide matrix

In [41]:
if 'machine' in df_historical_production.columns:
    df_historical_production = df_historical_production.rename(columns={'machine': 'id'})

df_historical_production = df_historical_production.pivot(index='id', columns='day', values='production')
df_historical_production['total'] = df_historical_production.values[:, sum(df_historical_production.dtypes == 'int64') :].sum(axis=1) # safely sums production days
df_historical_production.head()

day,Day-01,Day-02,Day-03,Day-04,Day-05,Day-06,Day-07,Day-08,Day-09,Day-10,...,Day-12,Day-13,Day-14,Day-15,Day-16,Day-17,Day-18,Day-19,Day-20,total
id,,,,,,,,,,,,,,,,,,,,,
M-01,106,104,130,130,130,130,104,104,102,130,...,130,130,122,120,130,110,130,105,100,0
M-02,56,64,57,56,73,80,78,81,76,82,...,84,79,77,63,64,65,72,79,65,0
M-03,116,140,138,118,140,140,127,108,110,140,...,123,132,107,140,126,137,124,134,121,0
M-04,86,105,84,83,111,84,97,95,92,116,...,90,114,87,85,100,109,102,109,81,0
M-05,89,85,78,87,102,99,108,111,81,109,...,86,89,88,86,107,76,97,105,89,0


[J] ln6 og and pg are aligned including out6 - this bc og ln5 had that exemplrary error at ln5 as mentioned above 

[og] Next load the historical failure data for these machines.

The column 'mid' represents the number of days before failure, according to historical records.

[pg] Note: Using that if check to catch whether the column is named 'machine' or 'id' is a great developer habit—it saves you from those annoying KeyError crashes we talked about earlier!

What to Expect Next  
When you run this cell, your output will transform the vertical list of production records into a clean matrix grid where each machine (M-01, M-02, etc.) has its daily output spread across columns like Day-01, Day-02, and a final total sum column. 

[pg]2 your pivot transformation has been executed cleanly.

That newly generated total column at the end is doing vital heavy lifting: it aggregates the cumulative production output across all days for each machine, giving our machine learning model a clear macro-view of how hard each asset was pushed.

📊 Step 4: Loading Historical Failure Data

Now that our production matrix is structured, we need the final piece of historical ground truth: when did the machines actually break?  

In the original IBM notebook, this dataset is loaded into a dataframe named df_historical_mid. The column mid represents the actual recorded number of days before failure happened historically. Comparing this against the vendor's initial specifications reveals the real-world performance drift our model needs to learn.  

Add the Code Cell
Create a new code cell in your predictive_maintenance.ipynb notebook, paste the code below, and run it (Shift + Enter):  


In [42]:
df_historical_mid = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/historical-mid.csv')
df_historical_mid.columns = ['id', 'mid']
df_historical_mid = df_historical_mid.set_index(['id'])
df_historical_mid.head()

,mid
id,
M-01,5
M-02,7
M-03,12
M-04,12
M-05,13


[pg] System Architecture Insight: Just like we did with the machine specs, we grab the raw CSV from the public repository, cleanly rename the columns to match standard identifiers (id and mid), and index it by machine ID so we can merge everything together cleanly.  

In short, mid stands for the actual number of days before failure, according to historical records.  While the vendor's specification gives us a theoretical estimate (remaining life), mid represents what actually happened in past factory data. We use it as our ground-truth target data so our machine learning model can learn how real-world conditions deviate from manufacturer claims. 

We pulled down a dataset representing past operational logs. In systems terms, think of historical-mid.csv as a black box flight recorder or crash log database. It records what actually happened to past factory machines when they were pushed to their limits.

1. The Source of mid
When we ran this line of code:
Python
df_historical_mid = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/historical-mid.csv')

2. Vendor Spec vs. Real-World Ground Truth
* remaining life: The manufacturer's theoretical, ideal-world estimate of how long a machine should last. 
* mid: The actual, real-world number of days the machine survived before failure under harsh factory conditions. 
Because machines face different workloads, heat, and production pressure, their real-world lifespan (mid) almost always deviates from the vendor's marketing claims.

3. How We Use mid in Machine Learning
In our machine learning model, mid plays a very specific role: it serves as our target label (the "answer key").

When we train our Scikit-Learn linear regression model:
* The Features (X): We feed the model the vendor's life estimate and the daily production volume. 
* The Target (y): We point the model to the mid column (y_train = df_historical.iloc[:, -1]). 
The regression algorithm analyzes the patterns—such as "When a machine with a vendor life of 15 days is forced to produce 130 units a day, it usually breaks in 12 days instead"—and learns to predict mid for new machines that haven't failed yet.


Does that clear up where mid comes from and why it is the most critical piece of ground-truth data in our predictive engine, Joshwin?

Next, we will merge all these data streams into a single master training table for our regression model.

so if you had to analogize some aspects in F(x) = y terms, how would it be?

That is the ultimate developer way to look at it, Joshwin. When you strip away all the corporate data science jargon, every machine learning model is just a function taking inputs and mapping them to an output.

Let's translate everything we just built into strict $f(x) = y$ terms.

The Grand Function:

$$f(x) = y$$

In our predictive maintenance pipeline, here is how those mathematical slots are filled:

* The Input Vector ($x$): This is everything you feed into the model. In our code, $x$ isn't just a single number; it's a multi-dimensional bundle of features. It includes the vendor's theoretical lifespan (life) combined with every single daily production volume column (Day-01 through Day-20). 
* The Function ($f$): This is our Scikit-Learn Linear Regression model (regr = linear_model.
LinearRegression()). During training (regr.fit(X_train, y_train)), the algorithm acts as the master formula designer, calculating the precise mathematical weights (coefficients) that connect heavy daily production workloads to premature machine failure. 
* The Output ($y$): This is your target prediction—the single, most probable real-world failure day, which we call mid. 

Translating It to Your World (The Systems Analogy)

If you think about it like game development or system modding:
* $x$ is your character's raw inventory and equipment stats (e.g., armor weight, durability, and how hard you are pushing your hardware).
* $f$ is the game engine's internal damage formula or physics calculation.
* $y$ is the exact frame or second your weapon breaks or your system crashes under stress.

The machine learning algorithm (f) doesn't magically guess the outcome; it reverse-engineers the formula by looking at historical crash logs so it can accurately predict $y$ for every new machine you load in.
Does viewing the pipeline through this mathematical lens make the code structure click into place for you?


[jpg] yes also, I liked the previous explanation as well the one with black box analogies etc. Try layer explanations with all the aspects so I can add depth to my comprehension from now on.

[pg]Deal, Joshwin. From here on out, every major concept we tackle will be broken down through our four-layer lens:
1. The Code Syntax (How it's written in Python)
2. The Systems / Game Dev Analogy (Relating it to architecture, file structures, or game mechanics you already intuitively understand)
3. The Mathematical Model (f(x)=y) (Mapping data flow to inputs, functions, and outputs)
4. The Real-World Domain Context (Why this matters in business, science, or engineering)
Now that we have successfully loaded our machine specs, production logs, and failure logs as separate files, it is time to do the next crucial step: Merging the Data Streams and Training the Linear Regression Model.
Step 5: Merging the DataFrames into a Single Master Table
Before our model can learn anything, all those separate tables (historical-machine, historical-production, and historical-mid) need to be joined together using the machine id as the primary key.
1. The Code Syntax
Create a new cell in your predictive_maintenance.ipynb notebook, paste this code, and run it (Shift + Enter):
Python

df_historical = df_historical_machine.join(df_historical_production).join(df_historical_mid)
df_historical.head()
2. The Multi-Layered Breakdown
* Systems / Game Dev Analogy: Think of this like a relational database join or merging separate mod configuration files. You have a file for character base stats (machine), a file for action logs (production), and a file for death events (mid). The machine id acts as the universal Primary Key / ID tag that binds them all to the exact same entity instance.
* Mathematical Perspective (f(x)=y): This step ensures that your multi-dimensional input variables (x) and your exact target label (y) live on the exact same row for every single machine. Without this merge, your inputs and outputs are floating around in isolated data silos.
* Real-World Context: In enterprise data engineering, data is always siloed across different departments (sales data is in one database, maintenance logs in another). Merging them into a single "master record view" is the foundational step of any predictive analytics pipeline.
Step 6: Splitting Features (X) and Target (y) & Training the Model
Now that our master table is assembled, we separate our inputs (X) from our ground-truth answer (y) and hand them over to Scikit-Learn to train the linear regression algorithm.
1. The Code Syntax
Create your next notebook cell, paste the code below, and run it (Shift + Enter):
Python

# Separate features (X) from the target label (y)
X_train = df_historical.iloc[:, :-1]  # Every column except the last one (specs + production days)
y_train = df_historical.iloc[:, -1]   # The final column: actual failure day (mid)

# Initialize and train the Linear Regression model
regr = linear_model.LinearRegression()
regr.fit(X_train, y_train)

print("Model training complete! Regression weights calculated.")
2. The Multi-Layered Breakdown
* Systems / Game Dev Analogy: Think of regr.fit() like running a machine learning compilation script. You are feeding historical playtest telemetry into an empty rules engine. The engine runs through all 20 machines, analyzes how production load affects survival time, and writes out the final compiled "damage formula" coefficients.
* Mathematical Perspective (f(x)=y):
    * X_train contains your multi-variable input matrix (x1 ,x2 ,x3 …xn ).
    * y_train contains the actual observed output vector (y).
    * The regr.fit() function solves the linear equation to find the optimal weights (coefficients and intercept) that minimize the error between prediction and reality.
* Real-World Context: This is where raw data officially transitions into "intelligence." The model is no longer just looking at past history; it is extracting a generalized mathematical rule that it can apply to future machines that haven't been built or broken yet.
Run those two code blocks, Joshwin, and let me know what confirmation message prints out on your screen!


[pg] Step 5: Merging the DataFrames into a Single Master Table

Before our model can learn anything, all those separate tables (historical-machine, historical-production, and historical-mid) need to be joined together using the machine id as the primary key.

1. The Code Syntax
Create a new cell in your predictive_maintenance.ipynb notebook, paste this code, and run it (Shift + Enter):
Python

df_historical = df_historical_machine.join(df_historical_production).join(df_historical_mid)
df_historical.head()

2. The Multi-Layered Breakdown
* Systems / Game Dev Analogy: Think of this like a relational database join or merging separate mod configuration files. You have a file for character base stats (machine), a file for action logs (production), and a file for death events (mid). The machine id acts as the universal Primary Key / ID tag that binds them all to the exact same entity instance.
* Mathematical Perspective ($f(x) = y$): This step ensures that your multi-dimensional input variables ($x$) and your exact target label ($y$) live on the exact same row for every single machine. Without this merge, your inputs and outputs are floating around in isolated data silos.
* Real-World Context: In enterprise data engineering, data is always siloed across different departments (sales data is in one database, maintenance logs in another). Merging them into a single "master record view" is the foundational step of any predictive analytics pipeline.


[j] note: that from the next step onwards the code syntax and superflous explanations will be cut down.

In [43]:
df_historical = df_historical_machine.join(df_historical_production).join(df_historical_mid)
df_historical.head()

,life,Day-01,Day-02,Day-03,Day-04,Day-05,Day-06,Day-07,Day-08,Day-09,...,Day-13,Day-14,Day-15,Day-16,Day-17,Day-18,Day-19,Day-20,total,mid
id,,,,,,,,,,,,,,,,,,,,,
M-01,5,106,104,130,130,130,130,104,104,102,...,130,122,120,130,110,130,105,100,0,5
M-02,9,56,64,57,56,73,80,78,81,76,...,79,77,63,64,65,72,79,65,0,7
M-03,14,116,140,138,118,140,140,127,108,110,...,132,107,140,126,137,124,134,121,0,12
M-04,13,86,105,84,83,111,84,97,95,92,...,114,87,85,100,109,102,109,81,0,12
M-05,15,89,85,78,87,102,99,108,111,81,...,89,88,86,107,76,97,105,89,0,13


[og] Comparing the remaining life given by the vendor with the historical failure, you can see that there is indeed a significant difference which would be valuable to predict.

[j] pg initailly skipped this next step because:
 "In software development, code blocks generally fall into two buckets: functional logic (the code that actually computes, transforms data, or trains models) and diagnostic/inspection logic (code that just prints text to the screen for a human to review). I mistakenly treated that print loop as an optional debugging step rather than an essential component of the notebook's layout."

[pg]That specific print loop is indeed part of the original IBM notebook sequence right after merging the historical data frames. It serves as a manual validation step to inspect the records before passing them into the machine learning model.

Running the Validation Loop

In [44]:
for m in range(1, 5):
    id = "M-0" + str(m)
    print(id)
    print("Remaining life for ", id, ": ", df_historical.life[id])
    print("Historical failure for ", id, ": ", df_historical.mid[id])

M-01
Remaining life for  M-01 :  5
Historical failure for  M-01 :  5
M-02
Remaining life for  M-02 :  9
Historical failure for  M-02 :  7
M-03
Remaining life for  M-03 :  14
Historical failure for  M-03 :  12
M-04
Remaining life for  M-04 :  13
Historical failure for  M-04 :  12


[pg] What this code does in system terms:
* The Iteration Loop (for m in range(1, 5):): This is a standard programming control flow loop that cycles through numbers 1 through 4.

* String Concatenation (id = "M-0" + str(m)): It dynamically stitches together our asset primary keys (M-01, M-02, M-03, M-04), acting much like generating file paths or asset IDs on the fly in software development.

* Manual Assertion Check (print(...)): In game development or systems engineering, when you patch or merge large databases, you always write quick debug loops to query specific keys and verify that your join or payload didn't scramble or drop values. This loop prints out the vendor's theoretical life side-by-side with the actual historical mid failure day so you can instantly spot the operational drift.

[j] perceived error from last code from [pg], will try running revised code

[pg]Step-by-Step: Forcing the 20-Machine Table to Display
Create a brand new code cell right below your current working area in predictive_maintenance.ipynb, paste the snippet below, and run it (Shift + Enter):

In [45]:
# Explicitly verify our downscaled master dataframe dimensions and rows
print("Total shape of our downscaled dataset (Rows, Columns):", df_historical.shape)

# Display the full 20-machine table preview
df_historical.head(20)

Total shape of our downscaled dataset (Rows, Columns): (1000, 23)


,life,Day-01,Day-02,Day-03,Day-04,Day-05,Day-06,Day-07,Day-08,Day-09,...,Day-13,Day-14,Day-15,Day-16,Day-17,Day-18,Day-19,Day-20,total,mid
id,,,,,,,,,,,,,,,,,,,,,
M-01,5,106,104,130,130,130,130,104,104,102,...,130,122,120,130,110,130,105,100,0,5
M-02,9,56,64,57,56,73,80,78,81,76,...,79,77,63,64,65,72,79,65,0,7
M-03,14,116,140,138,118,140,140,127,108,110,...,132,107,140,126,137,124,134,121,0,12
M-04,13,86,105,84,83,111,84,97,95,92,...,114,87,85,100,109,102,109,81,0,12
M-05,15,89,85,78,87,102,99,108,111,81,...,89,88,86,107,76,97,105,89,0,13
M-06,15,104,113,136,119,122,124,93,133,93,...,112,108,132,126,95,106,114,106,0,13
M-07,5,98,130,130,122,130,108,101,106,128,...,118,109,130,124,121,123,130,121,0,3
M-08,15,70,66,59,52,74,76,72,54,78,...,67,62,70,79,75,57,59,60,0,8
M-09,11,120,88,88,120,120,115,109,104,120,...,120,89,115,102,91,120,120,110,0,6


[og]The following is a chart that represents the deviation between the remaining life prediction from the vendor and the real failure, due to production trend.

[j] pg initially skipped this step

[pg]

In [46]:
import altair as alt

brush = alt.selection_interval()
alt.Chart(df_historical).mark_point(filled=True).encode(
    x='life:O',
    y='mid:Q'
).properties(
    width=800,
    height=300
)

alt.Chart(...)

In [47]:
import altair as alt

brush = alt.selection_interval()
alt.Chart(df_historical).mark_point(filled=True).encode(
    x='life:O',
    y='mid:Q'
).properties(
    width=800,
    height=300
)

alt.Chart(...)

In [48]:
import altair as alt

# Enable native VS Code mimetype rendering
alt.renderers.enable('mimetype')

brush = alt.selection_interval()
alt.Chart(df_historical).mark_point(filled=True).encode(
    x='life:O',
    y='mid:Q'
).properties(
    width=800,
    height=300
)

<VegaLite 6 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [49]:
import altair as alt

# Revert to the default renderer
alt.renderers.enable('default')

brush = alt.selection_interval()
alt.Chart(df_historical).mark_point(filled=True).encode(
    x='life:O',
    y='mid:Q'
).properties(
    width=800,
    height=300
)

alt.Chart(...)

[j] issues solved with altair with help of pg

[og] Failure prediction model training

You now can train a simple linear regression model to predict the failure using vendor's remaining life indication and the planned production for the machine as features.

[pg]1. Training the Linear Regression Model
With your historical dataset cleaned and merged, you can now train a Scikit-Learn linear regression model. This model learns the relationship between the vendor's estimated remaining life combined with planned daily production features (X_train) and the actual historical failure day (y_train).


In [50]:
from sklearn import linear_model

# Separate features (everything except the last column) and target (the last column: 'mid')
X_train = df_historical.iloc[:, :-1]
y_train = df_historical.iloc[:, -1]

# Create linear regression object
regr = linear_model.LinearRegression()

# Train the model using the historical training sets
regr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](22,)","[ 0.79,-0.01, 0.01,..., 0.01, 0.01, 0. ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](22,)","['life','Day-01','Day-02',...,'Day-19','Day-20','total']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-1.089
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,22
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(21)


[pg] 2. Applying the Trained Model to New Operational DataOnce the model is trained, you can apply it to your new operational dataset (df_machine) to forecast exact failure days for incoming active machines.  

****** restarting kernel after unpackingCPLEX - new code from [pg] fixing the nameerror from df_machine_x and df planned production x 

In [55]:
df_machine = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/machine.csv')
df_machine.head()

,id,capacity,remaining life,cost of maintenance,maintenance loss,cost of repair,repair loss,loss per life day,production value unit
0,M-01,130,14,50,50,100,100,20,10
1,M-02,120,13,50,50,100,100,20,10
2,M-03,130,9,50,50,100,100,20,10
3,M-04,110,10,50,50,100,100,20,10
4,M-05,110,17,50,50,100,100,20,10


In [56]:
# Keep only useful columns
df_machine_x = df_machine[['id', 'remaining life']];
df_machine_x.columns = ['id', 'life']
df_machine_x = df_machine_x.set_index(['id'])
df_machine_x.head()

,life
id,
M-01,14
M-02,13
M-03,9
M-04,10
M-05,17


[jnextday][jnd] Load the planned production and perform a simple transformation.

In [57]:
df_planned_production = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/planned_production.csv')
df_planned_production.columns = ['id', 'day', 'production']
df_planned_production.head()

,id,day,production
0,M-01,Day-01,103
1,M-01,Day-02,108
2,M-01,Day-03,105
3,M-01,Day-04,130
4,M-01,Day-05,130


In [58]:
df_planned_production_x = df_planned_production.pivot(index='id', columns='day', values='production')
df_planned_production_x['total'] = df_planned_production_x.values[:, 1:].sum(1)
df_planned_production_x.head()

day,Day-01,Day-02,Day-03,Day-04,Day-05,Day-06,Day-07,Day-08,Day-09,Day-10,...,Day-12,Day-13,Day-14,Day-15,Day-16,Day-17,Day-18,Day-19,Day-20,total
id,,,,,,,,,,,,,,,,,,,,,
M-01,103,108,105,130,130,130,104,130,120,115,...,129,102,130,122,130,120,119,110,114,2278
M-02,87,118,89,120,92,87,88,116,120,115,...,91,87,115,108,113,95,120,116,90,1973
M-03,130,115,113,130,121,130,130,126,121,130,...,108,130,130,114,130,130,130,130,130,2365
M-04,86,92,100,94,103,89,80,105,92,85,...,74,95,108,105,101,73,95,80,93,1759
M-05,93,83,73,73,93,104,108,88,92,86,...,96,104,74,94,102,93,83,77,87,1700


In [51]:
import os
print([f for f in os.listdir('.') if f.endswith('.csv')])

[]


In [52]:
import os

# Search recursively for any CSV files in your workspace
csv_files = []
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.csv'):
            csv_files.append(os.path.join(root, file))

print("Found CSV files:", csv_files)
print("Current working directory:", os.getcwd())

Found CSV files: ['./venv/lib/python3.14/site-packages/numpy/_core/tests/data/umath-validation-set-log2.csv', './venv/lib/python3.14/site-packages/numpy/_core/tests/data/umath-validation-set-arcsinh.csv', './venv/lib/python3.14/site-packages/numpy/_core/tests/data/umath-validation-set-arctanh.csv', './venv/lib/python3.14/site-packages/numpy/_core/tests/data/umath-validation-set-sin.csv', './venv/lib/python3.14/site-packages/numpy/_core/tests/data/umath-validation-set-cos.csv', './venv/lib/python3.14/site-packages/numpy/_core/tests/data/umath-validation-set-cbrt.csv', './venv/lib/python3.14/site-packages/numpy/_core/tests/data/umath-validation-set-arctan.csv', './venv/lib/python3.14/site-packages/numpy/_core/tests/data/umath-validation-set-cosh.csv', './venv/lib/python3.14/site-packages/numpy/_core/tests/data/umath-validation-set-expm1.csv', './venv/lib/python3.14/site-packages/numpy/_core/tests/data/umath-validation-set-sinh.csv', './venv/lib/python3.14/site-packages/numpy/_core/tests/

In [54]:
import pandas as pd
df_machine = pd.read_csv('name_of_machine_file.csv')
df_planned_production = pd.read_csv('name_of_production_file.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'name_of_machine_file.csv'

In [34]:
import pandas as pd

# Load the machine dataset locally (adjust path if your CSV is inside a data folder)
df_machine = pd.read_csv('machine.csv') 
df_planned_production = pd.read_csv('planned_production.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'machine.csv'

In [29]:
# Prepare the machine specifications for prediction
df_machine_x = df_machine[['id', 'life']].copy()
df_machine_x = df_machine_x.set_index(['id'])

# Reshape the planned production data for the new fleet
df_planned_production_x = df_planned_production.pivot(index='id', columns='day', values='production')
df_planned_production_x['total'] = df_planned_production_x.values[:, 1:].sum(axis=1)

# Now the join will work successfully
X_test = df_machine_x.join(df_planned_production_x)
X_test.head()

NameError: name 'df_machine' is not defined

In [22]:
# Combine machine specs and planned production into the test feature set
X_test = df_machine_x.join(df_planned_production_x)

# Predict the most probable failure day ('mid') for the new fleet
y_pred = regr.predict(X_test)

# Append predictions back into your dataframe
X_test['mid'] = y_pred
X_test.head()

NameError: name 'df_machine_x' is not defined

In [33]:
# Load planned production for the new machines
df_planned_production = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/production.csv')

# Pivot and compute totals if needed, then combine with machine characteristics
df_planned_production_x = df_planned_production.pivot(index='id', columns='day', values='production')

# Create the final test feature set by joining machine specs and production schedule
X_test = df_machine_x.join(df_planned_production_x)

# Predict the failure using the linear regression model
y_pred = regr.predict(X_test)

# Store predictions back into a dataframe for the next steps
df_prediction = pd.DataFrame({'id': X_test.index, 'predicted_mid': y_pred}).set_index('id')
df_prediction.head()

HTTPError: HTTP Error 404: Not Found

[j] just directly pasting og code from now on till any errors pop up

In [19]:
df_machine = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/machine.csv')
df_machine.head()

,id,capacity,remaining life,cost of maintenance,maintenance loss,cost of repair,repair loss,loss per life day,production value unit
0,M-01,130,14,50,50,100,100,20,10
1,M-02,120,13,50,50,100,100,20,10
2,M-03,130,9,50,50,100,100,20,10
3,M-04,110,10,50,50,100,100,20,10
4,M-05,110,17,50,50,100,100,20,10


In [20]:
# Keep only useful columns
df_machine_x = df_machine[['id', 'remaining life']];
df_machine_x.columns = ['id', 'life']
df_machine_x = df_machine_x.set_index(['id'])
df_machine_x.head()

,life
id,
M-01,14
M-02,13
M-03,9
M-04,10
M-05,17


*** [j] [og] [jog] note: ln[20] j = ln[13] og

In [21]:
df_planned_production = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/planned_production.csv')
df_planned_production.columns = ['id', 'day', 'production']
df_planned_production.head()

,id,day,production
0,M-01,Day-01,103
1,M-01,Day-02,108
2,M-01,Day-03,105
3,M-01,Day-04,130
4,M-01,Day-05,130


In [22]:
df_planned_production_x = df_planned_production.pivot(index='id', columns='day', values='production')
df_planned_production_x['total'] = df_planned_production_x.values[:, 1:].sum(1)
df_planned_production_x.head()

day,Day-01,Day-02,Day-03,Day-04,Day-05,Day-06,Day-07,Day-08,Day-09,Day-10,...,Day-12,Day-13,Day-14,Day-15,Day-16,Day-17,Day-18,Day-19,Day-20,total
id,,,,,,,,,,,,,,,,,,,,,
M-01,103,108,105,130,130,130,104,130,120,115,...,129,102,130,122,130,120,119,110,114,2278
M-02,87,118,89,120,92,87,88,116,120,115,...,91,87,115,108,113,95,120,116,90,1973
M-03,130,115,113,130,121,130,130,126,121,130,...,108,130,130,114,130,130,130,130,130,2365
M-04,86,92,100,94,103,89,80,105,92,85,...,74,95,108,105,101,73,95,80,93,1759
M-05,93,83,73,73,93,104,108,88,92,86,...,96,104,74,94,102,93,83,77,87,1700


In [59]:
X_test = df_machine_x.join(df_planned_production_x)
X_test.head()

,life,Day-01,Day-02,Day-03,Day-04,Day-05,Day-06,Day-07,Day-08,Day-09,...,Day-12,Day-13,Day-14,Day-15,Day-16,Day-17,Day-18,Day-19,Day-20,total
id,,,,,,,,,,,,,,,,,,,,,
M-01,14,103,108,105,130,130,130,104,130,120,...,129,102,130,122,130,120,119,110,114,2278
M-02,13,87,118,89,120,92,87,88,116,120,...,91,87,115,108,113,95,120,116,90,1973
M-03,9,130,115,113,130,121,130,130,126,121,...,108,130,130,114,130,130,130,130,130,2365
M-04,10,86,92,100,94,103,89,80,105,92,...,74,95,108,105,101,73,95,80,93,1759
M-05,17,93,83,73,73,93,104,108,88,92,...,96,104,74,94,102,93,83,77,87,1700


In [24]:
y_pred = regr.predict(X_test)

In [25]:
X_test['mid'] = y_pred
X_test.head()

,life,Day-01,Day-02,Day-03,Day-04,Day-05,Day-06,Day-07,Day-08,Day-09,...,Day-13,Day-14,Day-15,Day-16,Day-17,Day-18,Day-19,Day-20,total,mid
id,,,,,,,,,,,,,,,,,,,,,
M-01,14,103,108,105,130,130,130,104,130,120,...,102,130,122,130,120,119,110,114,2278,10.045138
M-02,13,87,118,89,120,92,87,88,116,120,...,87,115,108,113,95,120,116,90,1973,9.844368
M-03,9,130,115,113,130,121,130,130,126,121,...,130,130,114,130,130,130,130,130,2365,6.320415
M-04,10,86,92,100,94,103,89,80,105,92,...,95,108,105,101,73,95,80,93,1759,7.265803
M-05,17,93,83,73,73,93,104,108,88,92,...,104,74,94,102,93,83,77,87,1700,11.722816


In [26]:
brush = alt.selection_interval()
alt.Chart(X_test).mark_point(filled=True).encode(
    x='life:O',
    y='mid:Q'
).properties(
    width=800,
    height=300
)


alt.Chart(...)

*** [og] Prepare predictions for optimization ***

In [60]:
df_day = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/day.csv')
df_day.head()

,id
0,Day-01
1,Day-02
2,Day-03
3,Day-04
4,Day-05


In [61]:
import random
import numpy as np

n_days = df_day['id'].count()

data_failure = []
for machine in df_machine['id']:
    life = int(df_machine[df_machine.id==machine]['remaining life'])
    capacity = int(df_machine[df_machine.id==machine]['capacity'])
    base = random.randint(int(0.7*capacity), capacity)    
    x = [life]
    
    mid = int(X_test.mid[machine])

    #print (str(x) + " --> " + str(mid))
    
    spread = random.randint(2, 6)
    #print spread
    n = 1000
    #s = np.random.poisson(mid, n)
    s = np.random.normal(mid, spread/4., n)

    #print s
    data = [0 for i in range(n_days)]
    for i, day in np.ndenumerate(df_day['id']):
        t = 0
        for j in range(1000):
            if int(s[j])==i[0]:
                t = t+1                    
        data_failure.append((machine, day, int (100.*t/n)))       
        
df_predicted_failure = pd.DataFrame(data=data_failure, columns=['machine', 'day', 'failure'])
df_predicted_failure.head()

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'Series'

In [62]:
import random
import numpy as np

n_days = df_day['id'].count()

data_failure = []
for machine in df_machine['id']:
    life = int(df_machine.loc[df_machine.id == machine, 'remaining life'].values[0])
    capacity = int(df_machine.loc[df_machine.id == machine, 'capacity'].values[0])
    base = random.randint(int(0.7*capacity), capacity)    
    x = [life]
    
    mid = int(X_test.mid[machine])

    #print (str(x) + " --> " + str(mid))
    
    spread = random.randint(2, 6)
    #print spread
    n = 1000
    #s = np.random.poisson(mid, n)
    s = np.random.normal(mid, spread/4., n)

    #print s
    data = [0 for i in range(n_days)]
    for i, day in np.ndenumerate(df_day['id']):
        t = 0
        for j in range(1000):
            if int(s[j])==i[0]:
                t = t+1                    
        data_failure.append((machine, day, int (100.*t/n)))       
        
df_predicted_failure = pd.DataFrame(data=data_failure, columns=['machine', 'day', 'failure'])
df_predicted_failure.head()


AttributeError: 'DataFrame' object has no attribute 'mid'

In [64]:
import random
import numpy as np

# Safety check: Ensure the regression prediction 'mid' column exists in X_test
if 'mid' not in X_test.columns:
    y_pred = regr.predict(X_test)
    X_test['mid'] = y_pred

n_days = df_day['id'].count()

data_failure = []
for machine in df_machine['id']:
    # Safe scalar extraction using .values[0]
    life = int(df_machine.loc[df_machine.id == machine, 'remaining life'].values[0])
    capacity = int(df_machine.loc[df_machine.id == machine, 'capacity'].values[0])
    
    base = random.randint(int(0.7 * capacity), capacity)    
    x = [life]
    
    # Safe lookup using .loc to grab the predicted failure day ('mid')
    mid = int(X_test.loc[machine, 'mid'])

    spread = random.randint(2, 6)
    n = 1000
    s = np.random.normal(mid, spread / 4., n)

    data = [0 for i in range(n_days)]
    for i, day in np.ndenumerate(df_day['id']):
        t = 0
        for j in range(1000):
            if int(s[j]) == i[0]:
                t = t + 1                    
        data_failure.append((machine, day, int(100. * t / n)))       
        
df_predicted_failure = pd.DataFrame(data=data_failure, columns=['machine', 'day', 'failure'])
df_predicted_failure.head()

,machine,day,failure
0,M-01,Day-01,0
1,M-01,Day-02,0
2,M-01,Day-03,0
3,M-01,Day-04,0
4,M-01,Day-05,0


ln 22 = ln 32 [j]

In [65]:
df_failure_m01 = df_predicted_failure[df_predicted_failure.machine == 'M-01']

alt.Chart(df_failure_m01).mark_trail().encode(
    x='day',
    y='failure',
).properties(
    width=800,
    height=300
)

alt.Chart(...)

In [66]:
df_predicted_failure.reset_index(inplace=True)
df_predicted_failure = df_predicted_failure.set_index(['machine', 'day'])

df_planned_production.rename(columns={'id':'machine'}, inplace=True)
df_planned_production.reset_index(inplace=True)
df_planned_production = df_planned_production.set_index(['machine', 'day'])

In [67]:
# first global collections to iterate upon
all_machines = df_machine['id'].values
all_days = df_day['id'].values

data_cumul_failure = []
for machine in all_machines:
    for i, d in np.ndenumerate(all_days):
        cumul = 0
        for i2, d2 in np.ndenumerate(all_days):
            if i2==i:
                break
            cumul += int(df_predicted_failure.failure[machine, d2])
        data_cumul_failure.append((machine, d, cumul))

df_cumul_failure = pd.DataFrame(data_cumul_failure, columns=['machine', 'day', 'cumul_failure'])
df_cumul_failure=df_cumul_failure.set_index(['machine', 'day'])
df_cumul_failure.head()

cumul_failure
machine day                  
M-01    Day-01              0
        Day-02              0
        Day-03              0
        Day-04              0
        Day-05              0

In [68]:
df_cumul = df_cumul_failure.reset_index()
df_cumul_m01 = df_cumul[df_cumul.machine == 'M-01']

alt.Chart(df_cumul_m01).mark_trail().encode(
    x='day',
    y='cumul_failure',
).properties(
    width=800,
    height=300
)

alt.Chart(...)

In [82]:
# Select the first 20 machine IDs to bypass CPLEX size restrictions
target_machines = df_machine['id'].head(20)

# Filter core machine and test feature sets
df_machine = df_machine[df_machine['id'].isin(target_machines)]
X_test = X_test.loc[X_test.index.isin(target_machines)]

# Safely filter df_predicted_failure (handles cases where 'machine' is an index)
if 'machine' not in df_predicted_failure.columns:
    df_predicted_failure = df_predicted_failure.reset_index()
df_predicted_failure = df_predicted_failure[df_predicted_failure['machine'].isin(target_machines)]

# Safely filter df_planned_production (handles both 'id' and 'machine' naming conventions)
if 'id' not in df_planned_production.columns and 'machine' not in df_planned_production.columns:
    df_planned_production = df_planned_production.reset_index()
prod_id_col = 'id' if 'id' in df_planned_production.columns else 'machine'
df_planned_production = df_planned_production[df_planned_production[prod_id_col].isin(target_machines)]

print(f"Successfully downscaled to {len(target_machines)} machines for local optimization.")

Successfully downscaled to 20 machines for local optimization.


In [85]:
s = mdl.solve(log_output=True)
assert s, "solve failed"
mdl.report()

Using size restricted mode (Could not find directory for cpxchecklic).
CPLEX Error  1016: Community Edition. Problem size limits exceeded. Purchase at http://ibm.biz/error1016.


DOcplexLimitsExceeded: **** Promotional version. Problem size limits (1000 vars, 1000 consts) exceeded, model has 1200 vars, 650 consts, CPLEX code=1016

In [86]:
# Select only 3 machine IDs to strictly stay under the 1,000 variable CPLEX limit
target_machines = df_machine['id'].head(3)

# Filter core machine and test feature sets
df_machine = df_machine[df_machine['id'].isin(target_machines)]
X_test = X_test.loc[X_test.index.isin(target_machines)]

# Safely filter df_predicted_failure
if 'machine' not in df_predicted_failure.columns:
    df_predicted_failure = df_predicted_failure.reset_index()
df_predicted_failure = df_predicted_failure[df_predicted_failure['machine'].isin(target_machines)]

# Safely filter df_planned_production
if 'id' not in df_planned_production.columns and 'machine' not in df_planned_production.columns:
    df_planned_production = df_planned_production.reset_index()
prod_id_col = 'id' if 'id' in df_planned_production.columns else 'machine'
df_planned_production = df_planned_production[df_planned_production[prod_id_col].isin(target_machines)]

print(f"Successfully downscaled to {len(target_machines)} machines for local optimization.")

Successfully downscaled to 3 machines for local optimization.


In [87]:
df_production = df_production.production.apply(lambda v: v.solution_value)
df_production.head()

DOcplexException: Model<PredictiveMaintenance> did not solve successfully

[og] Optimization of maintenance

Now you will create an optimization model to create the optimal maintenance plan, taking into account some constraints.

One last input data frame you need are the parameters.

In [79]:
# Select the first 20 machine IDs to bypass CPLEX size restrictions
target_machines = df_machine['id'].head(20)

# Filter all downstream dataframes to only include these 20 machines
df_machine = df_machine[df_machine['id'].isin(target_machines)]
X_test = X_test.loc[X_test.index.isin(target_machines)]
df_predicted_failure = df_predicted_failure[df_predicted_failure['machine'].isin(target_machines)]
df_planned_production = df_planned_production[df_planned_production['id'].isin(target_machines)]

print(f"Successfully downscaled to {len(target_machines)} machines for local optimization.")

KeyError: 'machine'

In [83]:
df_parameters = pd.read_csv('https://raw.githubusercontent.com/achabrier/data/master/parameters.csv')
df_parameters.head()

,id,value
0,strategy,1
1,maintenance crew size,2


In [84]:
# Select the first 20 machine IDs to bypass CPLEX size restrictions
target_machines = df_machine['id'].head(20)

# Filter all downstream dataframes to only include these 20 machines
df_machine = df_machine[df_machine['id'].isin(target_machines)]
X_test = X_test.loc[X_test.index.isin(target_machines)]
df_predicted_failure = df_predicted_failure[df_predicted_failure['machine'].isin(target_machines)]
df_planned_production = df_planned_production[df_planned_production['id'].isin(target_machines)]

print(f"Successfully downscaled to {len(target_machines)} machines for local optimization.")

KeyError: 'id'

You will use the docplex Python package to formulate the optimization model.

In [70]:
from docplex.mp.environment import Environment
env = Environment()
env.print_information()    

* system is: Darwin 64bit
* Python version 3.14.7, located at: /Users/joshwiin/Desktop/PredMain/venv/bin/python
* docplex is present, version is 2.32.264
* CPLEX library is present, version is 22.2.0.0, located at: /Users/joshwiin/Desktop/PredMain/venv/lib/python3.14/site-packages
* pandas is present, version is 3.0.5


[og] * CPLEX library is present, version is 22.1.1.0, located at: /opt/conda/envs/Python-RT23.1-Premium/lib/python3.10/site-packages

**** [j]

Create a new optimization model.

In [71]:
from docplex.mp.model import Model
mdl = Model(name="PredictiveMaintenance")

Create decision variables:

(real) production (taking into account maintenance or failures) per machine and day
maintenance per machine and day

In [72]:
production = mdl.continuous_var_matrix(keys1=all_machines, keys2=all_days, name=lambda ns: "Production_%s_%s" % (ns[0],ns[1]))
df_production = pd.DataFrame({'production': production})
df_production.index.names=['all_machines', 'all_days']

maintenance = mdl.binary_var_matrix(keys1=all_machines, keys2=all_days, name=lambda ns: "Maintenance_%s_%s" % (ns[0],ns[1]))
df_maintenance = pd.DataFrame({'maintenance': maintenance})
df_maintenance.index.names=['all_machines', 'all_days']

Add some constraints linking real production with planned production and maintenance.

[j] ln offset +1 due to pandas series error -> updated code from [pg] with "machine.loc" and values[0] -> insertions 

In [73]:
for machine in all_machines:       
    maintenance_loss = int(df_machine.loc[df_machine.id==machine]['maintenance loss'].values[0])/100.
    capacity = int(df_machine.loc[df_machine.id==machine]['capacity'].values[0])
    for day in all_days:   
        prod = df_planned_production.production[machine, day]
        #mdl.add_if_then( maintenance[machine, day] == 1, production[machine, day]== 0 )
        #mdl.add_if_then( maintenance[machine, day] == 0, production[machine, day]== df_production[df_production.machine==machine][df_production.day==day] )
        if (prod <= capacity*(1-maintenance_loss)):
            mdl.add_constraint( production[machine, day] == prod )
        else:
            mdl.add_constraint( production[machine, day] == prod - (prod-capacity*(1-maintenance_loss))*maintenance[machine, day])
        

Add other constraints:

* Perform exactly one maintenance per machine
* The number of maintenance jobs possible on the same day is limited by the crew size

In [74]:
# One maintenance per machine
for machine in all_machines:       
    mdl.add_constraint( mdl.sum(maintenance[machine, day] for day in all_days) == 1)
    
maintenance_crew_size = int(df_parameters.loc[df_parameters.id=='maintenance crew size']['value'].values[0])

# One maintenance at a time
for day in all_days:       
    mdl.add_constraint( mdl.sum(maintenance[machine, day] for machine in all_machines) <= maintenance_crew_size)

[j] same loc values error - offset +1
Create some cost structures to be used for objectives.


In [75]:
data_cost_maintenance = []
cost_kpis = []
# Cost of repair
for machine in all_machines:           
    #print machine
    life = int(df_machine.loc[df_machine.id==machine]['remaining life'].values[0])
    capacity = int(df_machine.loc[df_machine.id==machine]['capacity'].values[0])
    cost_of_maintenance = int(df_machine.loc[df_machine.id==machine]['cost of maintenance'].values[0])
    maintenance_loss = int(df_machine.loc[df_machine.id==machine]['maintenance loss'].values[0])/100.
    cost_of_repair = int(df_machine.loc[df_machine.id==machine]['cost of repair'].values[0])
    repair_loss = int(df_machine.loc[df_machine.id==machine]['repair loss'].values[0])/100.
    loss_per_life_day = int(df_machine.loc[df_machine.id==machine]['loss per life day'].values[0])
    production_value_unit = int(df_machine.loc[df_machine.id==machine]['production value unit'].values[0])
    
    previous_day = None
    for i, day in np.ndenumerate(all_days):
        cost = 0;
        prob_break_before = 0
        if (previous_day != None):
            prob_break_before = int(df_cumul_failure.cumul_failure[machine, previous_day])/100.
        previous_day = day
        
        #print prob_break_before
        
        # Cost of lost production if failure before maintenance
        for i2, day2 in np.ndenumerate(all_days):
            if (i2==i):
                break
            prob_break_day2 = int(df_predicted_failure.failure[machine, day2])/100.
            production_day2 = int(df_planned_production.production[machine, day2])
            if (production_day2 > capacity*(1-repair_loss)):
                cost += production_value_unit*prob_break_day2*(production_day2 - capacity*(1-repair_loss))
            
        # Cost of repair if breaking before maintenance
        cost += cost_of_repair*prob_break_before
        
        # Cost of maintenance
        cost += cost_of_maintenance*(1-prob_break_before)
        
        # Cost of lost production for maintenance
        production_day = int(df_planned_production.production[machine, day])
        if (production_day > capacity*(1-maintenance_loss)):
            cost += production_value_unit*(production_day - capacity*(1-maintenance_loss))
        
        # Cost of maintenance too early
        cost += loss_per_life_day*max(life-i[0], 0)
        
        #print cost
        data_cost_maintenance.append((machine, day, cost))
        
        cost_kpis.append(cost*maintenance[machine, day])
        
cost_kpi = mdl.sum(cost_kpis)
mdl.add_kpi(cost_kpi, "Cost")

df_cost_maintenance = pd.DataFrame(data_cost_maintenance, columns=['machine', 'day', 'cost_maintenance'])
#print df_cost_maintenance

total_planned_production = mdl.sum(df_planned_production.production)
mdl.add_kpi(total_planned_production, "Total Planned Production")
total_production = mdl.sum(df_production.production)
mdl.add_kpi(total_production, "Total Production")

DecisionKPI(name=Total Production,expr=Production_M-01_Day-01+Production_M-01_Day-02+Production_M-01_Da..)

* offest = -14 now (15 now)

Objective is depending of the strategy.

* with strategy 1, the expected cost is directly minimized
* with strategy 2, emulating some human decision-making, the maintenance are simply pushed near to the peak of failure probability.

In [76]:
# Extract the strategy parameter cleanly as an integer scalar value
strategy = int(df_parameters.loc[df_parameters.id == 'strategy', 'value'].values[0])

# Check which optimization objective/strategy to apply
if strategy == 1:
    # Strategy 1: Minimize standard maintenance and repair costs
    mdl.minimize(cost_kpi)
else:
    # Strategy 2: Minimize penalties based on timing (early vs. late maintenance)
    early = 10
    late = 1000
    temp = []     
    for machine in all_machines:           
        last_day = None
        for i, day in np.ndenumerate(all_days):
            last_day = day
            cumul_failure = int(df_cumul_failure.cumul_failure[machine, day])
            
            # If cumulative failure is greater than 0, apply a heavy penalty for being late
            if cumul_failure > 0:                            
                temp.append(late * maintenance[machine, day])
            else:
                # Otherwise, apply a smaller penalty proportional to how early it is (i[0] represents the index/day)
                temp.append(early * i[0] * maintenance[machine, day])
            
    late_kpi = mdl.sum(temp)
    mdl.minimize(late_kpi)

Print information on the model.

Even with this small didactic data set, the number of variables is higher than 1000 and hence the Commercial Edition of CPLEX needs to be used.

[pg] What this block is doing:
* The Strategy Switch: It looks up the optimization strategy type from your parameters table.

* Strategy 1 (cost_kpi): Focuses strictly on financial optimization, minimizing the direct costs of maintenance and breakdown repairs.

* Strategy 2 (late_kpi): Focuses on operational timing safety. It sets up a penalty system where performing maintenance after a failure (cumul_failure > 0) incurs a massive penalty (late = 1000), whereas doing it too early incurs a smaller, scaled penalty (early = 10 multiplied by the day index i[0]). It then tells the optimization model (mdl.minimize) to find the schedule that minimizes this total penalty.

In [77]:
mdl.print_information()

Model: PredictiveMaintenance
 - number of variables: 1200
   - binary=600, integer=0, continuous=600
 - number of constraints: 650
   - linear=650
 - parameters: defaults
 - objective: minimize
 - problem type is: MILP


You now can solve the model.

The engine log shows how fast the model is solved.

In [78]:
s = mdl.solve(log_output=True)
assert s, "solve failed"
mdl.report()

all_kpis = [(kp.name, kp.compute()) for kp in mdl.iter_kpis()]
df_kpis = pd.DataFrame(all_kpis, columns=['kpi', 'value'])

Using size restricted mode (Could not find directory for cpxchecklic).
CPLEX Error  1016: Community Edition. Problem size limits exceeded. Purchase at http://ibm.biz/error1016.


DOcplexLimitsExceeded: **** Promotional version. Problem size limits (1000 vars, 1000 consts) exceeded, model has 1200 vars, 650 consts, CPLEX code=1016